# MCMC Sampling: Metropolis–Hastings & Gibbs

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/mcmc-sampling)

We implement a random-walk Metropolis sampler for a Bayesian posterior, a Gibbs sampler for a correlated bivariate Gaussian, then the practical machinery: trace plots, burn-in, acceptance-rate tuning, R-hat across chains, and effective sample size.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — Metropolis–Hastings for a Bayesian posterior

Infer the mean $\theta$ of Gaussian data with a Gaussian prior. We sample from the **unnormalized** log-posterior $\log p(x\mid\theta) + \log p(\theta)$ — the evidence never appears.

In [ ]:
# Data: 30 points from N(2, 1). Prior on theta: N(0, 10^2).
data = rng.normal(2.0, 1.0, 30)

def log_posterior(theta, data, sigma=1.0, prior_sd=10.0):
    log_lik   = -0.5 * np.sum((data - theta)**2) / sigma**2
    log_prior = -0.5 * theta**2 / prior_sd**2
    return log_lik + log_prior

def metropolis(log_post, init, n_steps, proposal_sd, seed=0):
    rng_local = np.random.default_rng(seed)
    theta = init
    lp = log_post(theta)
    chain = np.empty(n_steps)
    accepts = 0
    for t in range(n_steps):
        cand = theta + rng_local.normal(0, proposal_sd)
        lp_cand = log_post(cand)
        # symmetric proposal -> ratio is just exp(lp_cand - lp)
        if np.log(rng_local.random()) < lp_cand - lp:
            theta, lp = cand, lp_cand
            accepts += 1
        chain[t] = theta
    return chain, accepts / n_steps

chain, acc = metropolis(lambda th: log_posterior(th, data), init=0.0,
                        n_steps=8000, proposal_sd=0.5)
burn = 1000
posterior = chain[burn:]
print(f"Acceptance rate     = {acc:.2f}")
print(f"Posterior mean theta= {posterior.mean():.3f}  (data mean = {data.mean():.3f})")
print(f"95% credible interval = {np.percentile(posterior, [2.5, 97.5]).round(3)}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(chain, color='#6366f1', lw=0.5)
ax[0].axvline(burn, color='#f87171', ls='--', label='end of burn-in')
ax[0].set_title('Trace plot (fuzzy caterpillar = good mixing)')
ax[0].set_xlabel('iteration'); ax[0].set_ylabel('theta'); ax[0].legend()

ax[1].hist(posterior, bins=40, color='#6366f1', alpha=0.85, density=True)
ax[1].axvline(posterior.mean(), color='#f59e0b', label='posterior mean')
ax[1].set_title('Posterior of theta'); ax[1].set_xlabel('theta'); ax[1].legend()
plt.tight_layout(); plt.show()

## 2 — Acceptance-rate tuning

Proposal too small → tiny steps, slow exploration. Too large → most proposals rejected. Target ~23–44%.

In [ ]:
for psd in [0.02, 0.1, 0.5, 2.0, 10.0]:
    ch, acc = metropolis(lambda th: log_posterior(th, data), 0.0, 5000, psd)
    print(f"proposal_sd={psd:>5}:  acceptance={acc:.2f}   posterior_sd_est={ch[1000:].std():.3f}")
print("\nMid-range proposal_sd gives a healthy acceptance rate and the right posterior spread.")

## 3 — Gibbs sampling for a correlated bivariate Gaussian

The full conditional of a bivariate normal is 1-D Gaussian, so every Gibbs move is accepted ($\alpha=1$).

In [ ]:
rho = 0.9   # strong correlation
def gibbs_bivariate(n_steps, rho, seed=1):
    rng_local = np.random.default_rng(seed)
    x, y = 0.0, 0.0
    out = np.empty((n_steps, 2))
    cond_sd = np.sqrt(1 - rho**2)
    for t in range(n_steps):
        x = rng_local.normal(rho * y, cond_sd)   # x | y
        y = rng_local.normal(rho * x, cond_sd)   # y | x
        out[t] = (x, y)
    return out

samples = gibbs_bivariate(5000, rho)[500:]
print(f"Empirical correlation = {np.corrcoef(samples.T)[0,1]:.3f}  (target {rho})")

plt.figure(figsize=(5, 5))
plt.scatter(samples[:,0], samples[:,1], s=4, alpha=0.3, color='#6366f1')
plt.title(f'Gibbs samples, rho={rho}'); plt.xlabel('x'); plt.ylabel('y')
plt.tight_layout(); plt.show()

## 4 — R-hat and effective sample size

Run multiple chains from dispersed starts; R-hat compares between- vs within-chain variance. ESS discounts autocorrelation.

In [ ]:
def gelman_rubin(chains):
    """chains: (m, n) array of m chains each length n. Returns R-hat."""
    m, n = chains.shape
    chain_means = chains.mean(axis=1)
    B = n * chain_means.var(ddof=1)                  # between-chain
    W = chains.var(axis=1, ddof=1).mean()            # within-chain
    var_hat = (n - 1)/n * W + B/n
    return np.sqrt(var_hat / W)

def effective_sample_size(x, max_lag=200):
    x = x - x.mean()
    n = len(x)
    var = np.dot(x, x) / n
    ess_sum = 0.0
    for k in range(1, min(max_lag, n)):
        rho_k = np.dot(x[:-k], x[k:]) / (n * var)
        if rho_k < 0:   # standard truncation when autocorrelation goes negative
            break
        ess_sum += rho_k
    return n / (1 + 2 * ess_sum)

# 4 dispersed chains for theta
starts = [-8, -3, 3, 8]
chains = np.array([metropolis(lambda th: log_posterior(th, data), s, 5000, 0.5, seed=i)[1000:]
                   for i, s in enumerate(starts)])
print(f"R-hat = {gelman_rubin(chains):.4f}  (want < 1.01)")
for i, c in enumerate(chains):
    print(f"  chain {i}: length={len(c)}  ESS={effective_sample_size(c):.0f}")

## ✏️ Your turn

**Task A — Metropolis–Hastings for a Beta posterior:** Sample the posterior of a coin's bias `p` given `k` heads in `n` flips with a `Beta(2,2)` prior, using a random-walk proposal on the logit of `p` (so proposals stay in (0,1)). Compare the sample mean to the analytic posterior mean `(k+2)/(n+4)`.

**Task B — Diagnose a stuck chain:** Run the Metropolis sampler with `proposal_sd=0.001` and 4 chains. Show that the trace plots barely move, R-hat is well above 1, and ESS is tiny — the signature of a chain that hasn't converged.

In [ ]:
def sample_beta_posterior(k, n, n_steps=8000, proposal_sd=0.3, seed=0):
    """MH on logit(p) for a Beta(2,2) prior + Binomial likelihood."""
    def log_post(p):
        if p <= 0 or p >= 1:
            return -np.inf
        log_lik   = k*np.log(p) + (n-k)*np.log(1-p)
        log_prior = (2-1)*np.log(p) + (2-1)*np.log(1-p)   # Beta(2,2)
        return log_lik + log_prior
    # TODO(you): random-walk Metropolis on theta=logit(p); transform back with sigmoid;
    # return the chain of p values (after burn-in)
    return ...

k, n = 7, 10
chain_p = sample_beta_posterior(k, n)
if chain_p is not None:
    print(f"MH posterior mean p = {np.mean(chain_p):.4f}")
    print(f"Analytic mean       = {(k+2)/(n+4):.4f}")

<details><summary>Solution — Task A</summary>

```python
def sample_beta_posterior(k, n, n_steps=8000, proposal_sd=0.3, seed=0):
    rng_local = np.random.default_rng(seed)
    def log_post_logit(theta):
        p = 1/(1+np.exp(-theta))
        log_lik   = k*np.log(p) + (n-k)*np.log(1-p)
        log_prior = np.log(p) + np.log(1-p)             # Beta(2,2) kernel
        log_jac   = np.log(p) + np.log(1-p)             # logit change-of-variables
        return log_lik + log_prior + log_jac
    theta = 0.0
    lp = log_post_logit(theta)
    out = []
    for t in range(n_steps):
        cand = theta + rng_local.normal(0, proposal_sd)
        lpc = log_post_logit(cand)
        if np.log(rng_local.random()) < lpc - lp:
            theta, lp = cand, lpc
        out.append(1/(1+np.exp(-theta)))
    return np.array(out[1000:])
```

The Jacobian term makes the chain target the posterior in `p`-space correctly despite proposing in logit-space.
</details>